# 02 — Análisis Exploratorio de Datos (EDA) — Titanic

Este cuaderno sigue la arquitectura Medallion leyendo los artefactos generados por `flows/00_ingestion.py` (Bronze), `flows/01_data_engineering.py` (Silver) y `flows/03_aggregation.py` (Gold), sin rutas absolutas ni lógica productiva duplicada.


## Parte 1 — Primer contacto con los datos (Bronze)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from flows.common.paths import BRONZE, BRONZE_LABELED_NAME, GOLD, SILVER, SILVER_NAME

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 13

df_bronze = pd.read_csv(BRONZE / BRONZE_LABELED_NAME)
print("Dimensiones Bronze:", df_bronze.shape)


In [ ]:
# Paso 2 y 3: Estructura y primeras filas
df_bronze.info()
display(df_bronze.head(10))


In [ ]:
# Paso 4: Estadísticas descriptivas (numéricas y categóricas)
display(df_bronze.describe())
display(df_bronze.select_dtypes(exclude="number").describe())


In [ ]:
# Paso 5: Identificación de datos faltantes y mapa de calor
null_counts = df_bronze.isnull().sum()
display(null_counts[null_counts > 0])

plt.figure(figsize=(10, 4))
sns.heatmap(df_bronze.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Mapa de calor de valores faltantes en Bronze")
plt.show()


## Partes 2 a 5 — Exploración sobre capa Silver


In [ ]:
df = pd.read_parquet(SILVER / SILVER_NAME)
print("Dimensiones Silver:", df.shape)
display(df.head())


In [ ]:
# Paso 6: Distribución por género y cruce con clase
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Sex", ax=axes[0])
axes[0].set_title("Conteo por Sexo")

sns.countplot(data=df, x="Sex", hue="Pclass", ax=axes[1])
axes[1].set_title("Sexo por Clase (Pclass)")
plt.tight_layout()
plt.show()

display(pd.crosstab(df["Sex"], df["Pclass"], normalize="columns").round(3))


In [ ]:
# Paso 7, 8, 9 y 10: Edades reales (AgeOriginal), Person y KDE
plt.figure(figsize=(8, 4))
sns.histplot(df["AgeOriginal"].dropna(), bins=30, kde=True)
plt.title("Distribución de Edad Original (sin imputar)")
plt.xlabel("Edad (años)")
plt.show()

print(df["AgeOriginal"].agg(["mean", "median", "min", "max"]).round(2))
print("\nConteo por Person:")
display(df["Person"].value_counts())
display(pd.crosstab(df["Pclass"], df["Person"]))

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
sns.kdeplot(data=df, x="AgeOriginal", hue="Sex", fill=True, common_norm=False, ax=axes[0])
axes[0].set_title("Densidad de Edad por Sex")
sns.kdeplot(data=df, x="AgeOriginal", hue="Person", fill=True, common_norm=False, ax=axes[1])
axes[1].set_title("Densidad de Edad por Person")
sns.kdeplot(data=df, x="AgeOriginal", hue="Pclass", fill=True, common_norm=False, palette="viridis", ax=axes[2])
axes[2].set_title("Densidad de Edad por Pclass")
plt.tight_layout()
plt.show()


In [ ]:
# Pasos 11 a 15: Cubierta (TCabin), Puerto (Embarked) y Familia (Alone)
display(df["TCabin"].value_counts())

cubierta = df.loc[(df["TCabin"] != "X") & (df["TCabin"] != "T"), "TCabin"]
plt.figure(figsize=(7, 4))
sns.countplot(x=cubierta, order=sorted(cubierta.unique()))
plt.title("Distribución por Cubierta conocida (excluyendo T)")
plt.show()

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="Embarked", hue="Pclass")
plt.title("Puerto de embarque por Clase")
plt.show()
display(pd.crosstab(df["Embarked"], df["Pclass"], normalize="index").round(3))

print("\nProporción Alone:")
display(df["Alone"].value_counts(normalize=True).round(3))
sns.countplot(data=df, x="Alone")
plt.title("Pasajeros solos vs. con familia")
plt.show()


## Parte 4 — Análisis de supervivencia


In [ ]:
# Pasos 16 a 20: Supervivencia general, por clase, Person, edad y tarifa
print(f"Tasa general de supervivencia: {df['Survived'].mean():.3f}")

sns.catplot(data=df, x="Pclass", y="Survived", kind="point", height=4, aspect=1.4)
plt.title("Tasa de supervivencia por Clase")
plt.ylabel("Tasa de supervivencia")
plt.show()

sns.catplot(
    data=df, x="Pclass", y="Survived", hue="Person",
    hue_order=["child", "female", "male"], kind="point", height=4, aspect=1.5
)
plt.title("Tasa de supervivencia por Clase y Tipo de Persona")
plt.ylabel("Tasa de supervivencia")
plt.show()

generaciones = [10, 20, 40, 60, 80]
sns.lmplot(data=df, x="AgeOriginal", y="Survived", hue="Pclass", x_bins=generaciones, height=4.5, aspect=1.4)
plt.title("Supervivencia por rango de edad y clase")
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="AgeOriginal", y="Fare", hue="Pclass", alpha=0.6, palette="viridis")
plt.title("Relación Tarifa vs. Edad por Clase")
plt.show()


## Partes 5, 6 y 7 — Títulos, Correlaciones, Boxplots y Tarifa


In [ ]:
# Pasos 25-27: Títulos consolidados y matriz de correlación
display(pd.crosstab(df["TTitulo"], df["Sex"]))

cols = ["Survived", "Pclass", "NSex", "Age", "SibSp", "Parch", "Fare", "NEmbarked", "NCabin", "NTitulo"]
corr = df[cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Matriz de Correlación de Pearson (Silver)")
plt.show()

print("Correlación con Survived:")
display(corr["Survived"].sort_values())


In [ ]:
# Pasos 30 y 31: Boxplots 2x2 de AgeOriginal y distribución de Fare
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.boxplot(data=df, x="Sex", y="AgeOriginal", ax=axes[0, 0])
axes[0, 0].set_title("Edad por Sexo")
sns.boxplot(data=df, x="Pclass", y="AgeOriginal", ax=axes[0, 1])
axes[0, 1].set_title("Edad por Clase")
sns.boxplot(data=df, x="Embarked", y="AgeOriginal", ax=axes[1, 0])
axes[1, 0].set_title("Edad por Puerto")
sns.boxplot(data=df, x="Survived", y="AgeOriginal", ax=axes[1, 1])
axes[1, 1].set_title("Edad por Supervivencia")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(df["Fare"], kde=True, bins=40)
plt.title("Distribución de Tarifa (Fare)")
plt.show()
print(f"Fare -> Media: {df['Fare'].mean():.2f}, Mediana: {df['Fare'].median():.2f}, Asimetría (skew): {df['Fare'].skew():.2f}")


## Parte 9 — Análisis crítico y lectura de capa Gold


In [ ]:
import json

# Pasos 29 y 34-38: Tablas Gold, prueba Chi-cuadrado, tamaño familiar y atípicos
chi2_res = json.loads((GOLD / "chi2_sex_survival.json").read_text())
print("Prueba Chi-cuadrado Sex vs Survived:", chi2_res)

for name in [
    "survival_by_class_sex",
    "survival_by_embarked",
    "survival_by_embarked_class",
    "survival_by_fare_quartile_class",
    "outlier_summary",
]:
    print(f"\n--- {name} ---")
    display(pd.read_parquet(GOLD / f"{name}.parquet"))

plt.figure(figsize=(8, 4))
sns.pointplot(data=df, x="Familia", y="Survived")
plt.title("Supervivencia según tamaño de familia (con IC 95%)")
plt.show()

print("Pasajeros con tarifas más altas:")
display(df.nlargest(5, "Fare")[["PassengerId", "Name", "Pclass", "Sex", "AgeOriginal", "Fare", "Survived"]])
